# PetGuard AI — Final Application (Google Colab)

**Midterm → Final:** pretrained YOLO pet detection → end-user restricted zone → temporal confirmation → annotated video and event CSV.

Run the two setup cells once. **All zone changes, video uploads, and processing happen inside Gradio**; you do not rerun cells for Edit/Undo/Redraw/Confirm/Process.

**Known limitations:** fixed camera; sofa mode uses 2D overlap, not verified physical occupancy; human-as-dog false positives remain possible. See the evaluation notebook for ground-truth measurements.


## 1. Clone current GitHub code and install reproducible dependencies

In [ ]:
from pathlib import Path
import subprocess, sys, os

REPO = Path("/content/PetGuard-AI")
URL = "https://github.com/iamhuongthn/PetGuard-AI.git"
if REPO.exists() and (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "master"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "master"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/master"], check=True)
else:
    if REPO.exists():
        raise RuntimeError("Repository path exists but is not a Git clone.")
    subprocess.run(["git", "clone", "--branch", "master", URL, str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], check=True)
assert (REPO / "src/app.py").exists()
print("PetGuard source ready.")


## 2. Launch Gradio once; configure and reprocess in the browser

In [ ]:
import sys
sys.path.insert(0, str(REPO))
from src.app import build_app

app = build_app()
app.queue(default_concurrency_limit=1).launch(
    share=True, show_error=True, prevent_thread_lock=True
)


## 3. Validation and deliverables

**Test:** SAFE (no event), VIOLATION (event after 5 consecutive frames), human-only (no event expected; log actual failures), brief crossing (no confirmed event), zone Edit/Cancel/Redraw/reprocess, and no-zone processing guard.

The app writes each run under `results/runs/<run_id>/`. Download annotated MP4 and CSV before the Colab runtime ends. Use `PetGuard_Model_Evaluation.ipynb` to calculate accuracy from labeled samples. Model confidence and event count are **not** accuracy.

**Security:** Gradio's temporary public share link can be accessed by anyone who has it. Use only nonsensitive test videos.
